In [0]:
# Silver layer: products
# Source: b2b_ecommerce.bronze.products (append-only bronze ingestion)
# Grain:  one row per product_id and version (history via __START_AT/__END_AT)
# Approach: bronze.products is append-only, so we stream it, clean +
#           standardize each row, and merge into b2b_ecommerce.silver.products
#           with AUTO CDC (merge on product_id, sequenced by updated_at,
#           SCD Type 2 = keep the full change history).
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_PRODUCTS = "b2b_ecommerce.bronze.products"


def clean_text(name, lower=False, upper=False):
    """Trim a text column; a blank value ("" or " ") becomes NULL, not ""."""
    trimmed = F.trim(F.col(name))
    if lower:
        trimmed = F.lower(trimmed)
    if upper:
        trimmed = F.upper(trimmed)
    return (
        F.when(trimmed != "", trimmed)
        .otherwise(F.lit(None).cast("string"))
        .alias(name)
    )


# ---------------------------------------------------------------------------
# 1) Cleaned + standardized stream of bronze.products
# ---------------------------------------------------------------------------
@dp.temporary_view(name="products_clean_stream")
# Data quality validation (only checks that can actually catch bad rows).
# Drop-level: invalid rows are quarantined out of silver so downstream
# reads get a clean table (accuracy/validity enforced at the silver boundary).
@dp.expect_or_drop("product_id_not_null", "product_id IS NOT NULL")  # merge key must exist
@dp.expect_or_drop("updated_at_not_null", "updated_at IS NOT NULL")  # CDC sequence must exist; no created_at fallback
@dp.expect_or_drop("sku_not_null", "sku IS NOT NULL")
@dp.expect_or_drop("name_not_null", "name IS NOT NULL")
@dp.expect_or_drop("category_not_null", "category IS NOT NULL")
@dp.expect_or_drop("brand_not_null", "brand IS NOT NULL")
@dp.expect_or_drop("unit_of_measure_valid", "unit_of_measure IN ('set', 'unit', 'box', 'pack')")
@dp.expect_or_drop("list_price_positive", "list_price IS NOT NULL AND list_price > 0")
@dp.expect_or_drop("weight_kg_positive", "weight_kg IS NOT NULL AND weight_kg > 0")
@dp.expect_or_drop("min_order_qty_positive", "min_order_qty IS NOT NULL AND min_order_qty >= 1")
@dp.expect_or_drop("barcode_valid", "barcode IS NOT NULL AND barcode RLIKE '^[0-9]{13}$'")  # EAN-13
def products_clean_stream():
    # bronze.products is append-only: every change arrives as a new row.
    # The AUTO CDC merge below turns each change into a new version in the
    # SCD Type 2 history (sequenced by updated_at); re-emitted product_id
    # rows are deduplicated by the merge (latest updated_at wins).
    bronze = spark.readStream.table(BRONZE_PRODUCTS)

    return bronze.select(
        # Column types: explicit casts keep the silver schema stable
        F.col("product_id").cast("int").alias("product_id"),
        # Cleaning: trim; SKU is an identifier -> canonical UPPERCASE form;
        # a blank SKU becomes NULL (not "") so sku_not_null catches it
        clean_text("sku", upper=True),
        # Cleaning: trim whitespace; blank text -> NULL
        clean_text("name"),
        # Standardization: lowercase categories for consistent grouping
        clean_text("category", lower=True),
        F.col("list_price").cast("decimal(10,2)").alias("list_price"),
        # brand is a proper noun: trim only, keep original casing
        clean_text("brand"),
        clean_text("unit_of_measure", lower=True),
        F.col("weight_kg").cast("decimal(6,2)").alias("weight_kg"),
        clean_text("barcode"),
        F.col("is_active").cast("boolean").alias("is_active"),
        F.col("min_order_qty").cast("int").alias("min_order_qty"),
        F.col("created_at").cast("timestamp").alias("created_at"),
        # updated_at drives the CDC merge ordering. A row without it is
        # dropped by updated_at_not_null instead of falling back to
        # created_at — a fallback can order a late change before the
        # original version and corrupt the history.
        F.col("updated_at").cast("timestamp").alias("updated_at"),
    )


# ---------------------------------------------------------------------------
# 2) Silver target: b2b_ecommerce.silver.products
#    Rows are merged by product_id; when the same product is appended again
#    with a newer updated_at, a new version is kept (full history).
#    Current state per product = the row where __END_AT IS NULL.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="products",
    comment=(
        "Silver products: cleaned, standardized; full history per product_id "
        "with __START_AT/__END_AT validity (SCD Type 2)"
    ),
)

dp.create_auto_cdc_flow(
    target="products",
    source="products_clean_stream",
    keys=["product_id"],
    sequence_by="updated_at",
    stored_as_scd_type=2,
    # Only business columns open new versions: a re-sent row with a newer
    # updated_at but no real change updates in place instead of creating
    # a noisy duplicate version.
    track_history_except_column_list=["updated_at"],
)